# Results Table From Paths

In [ ]:
import json
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata

In [ ]:
PATHS = {
    # 'model-name': {
    #     'DatasetName': 'workdir/.../ue_manager_seed1',
    # },
}

paths_json = os.environ.get('RAUQ_PATHS_JSON')
if paths_json:
    source = Path(paths_json)
    PATHS = json.loads(source.read_text() if source.exists() else paths_json)

DATASET_METRICS = {
    'XSum': ['AlignScoreInv'],
    'XSUM': ['AlignScoreInv'],
    'SamSum': ['AlignScoreInv'],
    'CNN': ['AlignScoreInv'],
    'WMT14': ['Comet'],
    'WMT19': ['Comet'],
    'MedQUAD': ['AlignScore'],
    'TruthfulQA': ['AlignScore'],
    'CoQA': ['AlignScore'],
    'SciQ': ['AlignScore'],
    'TriviaQA': ['AlignScore'],
    'MMLU': ['Accuracy'],
    'GSM8k': ['Accuracy'],
}

GROUPS = {
    'QA': ['MedQUAD', 'TruthfulQA', 'CoQA', 'SciQ', 'TriviaQA', 'MMLU', 'GSM8k'],
    'Summ': ['XSum', 'XSUM', 'SamSum', 'CNN'],
    'MT': ['WMT14', 'WMT19'],
}

UE_METRIC = 'prr_0.5_normalized'
LEVEL = 'sequence'


In [ ]:
def is_instruction_model(model):
    model = str(model).lower()
    return 'instruct' in model or 'gpt-oss' in model


BASE_METHODS = [
    ('MSP', ['MaximumSequenceProbability']),
    ('Perplexity', ['Perplexity']),
    ('CCP', ['CCP']),
    ('Attention Score', [r're:LLMCheckAttention Layer \d+, sum']),
    ('Focus', [r're:Focus(?: .*)?']),
    ('Simple Focus', ['SimpleFocus reccurent']),
    ('DegMat NLI Score entail.', ['DegMat_NLI_score_entail']),
    ('Ecc. NLI Score entail.', ['Eccentricity_NLI_score_entail']),
    ('EVL NLI Score entail.', ['EigValLaplacian_NLI_score_entail']),
    ('Lexical Similarity Rouge-L', ['LexicalSimilarity_rougeL']),
    ('EigenScore', ['EigenScore sample_embeddings']),
    ('LUQ', ['LUQ (deberta)']),
    ('Semantic Entropy', ['SemanticEntropy']),
    ('SAR', ['SAR']),
    ('Semantic Density', ['SemanticDensity']),
]

INSTRUCT_METHODS = [
    ('MSP', ['MaximumSequenceProbability']),
    ('Perplexity', ['Perplexity']),
    ('MTE', ['MeanTokenEntropy']),
    ('CCP', ['CCP']),
    ('Attention Score', [r're:LLMCheckAttention Layer \d+, sum']),
    ('Simple Focus', ['SimpleFocus reccurent']),
    ('DegMat NLI Score entail.', ['DegMat_NLI_score_entail']),
    ('Ecc. NLI Score entail.', ['Eccentricity_NLI_score_entail']),
    ('EVL NLI Score entail.', ['EigValLaplacian_NLI_score_entail']),
    ('Lexical Similarity Rouge-L', ['LexicalSimilarity_rougeL']),
    ('EigenScore', ['EigenScore sample_embeddings']),
    ('LUQ', ['LUQ (deberta)']),
    ('Semantic Entropy', ['SemanticEntropy']),
    ('SAR', ['SAR']),
    ('Semantic Density', ['SemanticDensity']),
]

BASE_RAUQ_PROB_ALIASES = [
    r're:(?:RAUQ|UAD) max_meanlog_max 0\.20',
    r're:(?:RAUQ|UAD) max_meanlog_max 0\.2',
]

BASE_RAUQ_ENTROPY_ALIASES = [
    r're:(?:RAUQ|UAD) max_meanlog_max Entropy 0\.50',
    r're:(?:RAUQ|UAD) max_meanlog_max Entropy 0\.5',
]

INSTRUCT_RAUQ_PROB_ALIASES = [
    r're:(?:RAUQ|UAD) max_meanlog_max 0\.60',
    r're:(?:RAUQ|UAD) max_meanlog_max 0\.6',
]

INSTRUCT_RAUQ_ENTROPY_ALIASES = [
    r're:(?:RAUQ|UAD) max_meanlog_max Entropy 0\.90',
    r're:(?:RAUQ|UAD) max_meanlog_max Entropy 0\.9',
]


def paper_method_specs(model):
    if is_instruction_model(model):
        return INSTRUCT_METHODS + [('RAUQ', INSTRUCT_RAUQ_ENTROPY_ALIASES + INSTRUCT_RAUQ_PROB_ALIASES)]
    return BASE_METHODS + [('RAUQ', BASE_RAUQ_PROB_ALIASES + BASE_RAUQ_ENTROPY_ALIASES)]


In [ ]:
def load_manager(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return torch.load(path, weights_only=False)


def get_method_values(path, metric_names, dataset, ue_metric=UE_METRIC, level=LEVEL):
    managers = [load_manager(p) for p in path] if isinstance(path, list) else [load_manager(path)]
    values = {}

    for manager in managers:
        methods = sorted({key[1] for key in manager['metrics'] if key[0] == level and '2side' not in key[1]})
        for method in methods:
            for metric in metric_names:
                key = (level, method, metric, ue_metric)
                if key in manager['metrics']:
                    values.setdefault(method, []).append(manager['metrics'][key])

    if not values:
        raise ValueError(f'No metrics found for {dataset}: {metric_names}')

    return {method: np.mean(scores) for method, scores in values.items()}


In [ ]:
def matches_alias(method, alias):
    if alias.startswith('re:'):
        return re.fullmatch(alias[3:], method) is not None
    return method == alias


def score_for_aliases(raw_values, aliases):
    for alias in aliases:
        scores = [score for method, score in raw_values.items() if matches_alias(method, alias)]
        if scores:
            return np.mean(scores)
    return np.nan


def mean_rank(values):
    ranks = []
    for _, col in values.items():
        arr = col.to_numpy(dtype=float)
        mask = ~np.isnan(arr)
        col_ranks = np.full(len(arr), np.nan)
        if mask.any():
            col_ranks[mask] = rankdata(-arr[mask])
        ranks.append(col_ranks)
    return pd.Series(np.nanmean(np.vstack(ranks), axis=0), index=values.index).round(2)


def build_model_table(model, paths_by_dataset):
    specs = paper_method_specs(model)
    result = pd.DataFrame({('', 'Methods'): [name for name, _ in specs]})

    for dataset, path in paths_by_dataset.items():
        metric_names = DATASET_METRICS.get(dataset)
        if metric_names is None:
            raise KeyError(f'Add metric mapping for dataset {dataset!r}.')

        raw_values = get_method_values(path, metric_names=metric_names, dataset=dataset)
        result[(dataset, metric_names[0])] = [score_for_aliases(raw_values, aliases) for _, aliases in specs]

    value_cols = result.columns[1:]
    result[('', 'Mean')] = result[value_cols].mean(axis=1)
    result[('', 'Mean Rank')] = mean_rank(result[value_cols])
    return result


def group_table(model, df):
    grouped = pd.DataFrame({('', 'Methods'): df[('', 'Methods')]})
    for group, datasets in GROUPS.items():
        cols = [col for col in df.columns if col[0] in datasets]
        if cols:
            grouped[(model, group)] = df[cols].mean(axis=1)
    return grouped


def build_overall_table(paths):
    tables = []
    for i, (model, paths_by_dataset) in enumerate(paths.items()):
        model_table = group_table(model, build_model_table(model, paths_by_dataset))
        tables.append(model_table if i == 0 else model_table[model_table.columns[1:]])

    overall = pd.concat(tables, axis=1)
    value_cols = overall.columns[1:]
    overall[('', 'Mean')] = overall[value_cols].mean(axis=1)
    overall[('', 'Mean Rank')] = mean_rank(overall[value_cols])
    return overall.reset_index(drop=True)


In [ ]:
if not PATHS:
    raise ValueError('Fill PATHS with model -> dataset -> ue_manager_seed1 paths.')

table = build_overall_table(PATHS)
table.style.background_gradient(axis=0)